%md
# Petroleum Supply Analysis — End-to-End Pipeline

## 1. Load Shared Pipeline Components
Load reusable Bronze, Silver, and Gold functions.
Data processing begins only when the pipeline entry function is called.

In [0]:
%run ./Petroleum_Pipeline_Core

In [0]:
%run ./Petroleum_Gold_Core

In [0]:
required_functions = [
    "ingest_raw_batch_to_bronze",
    "process_bronze_to_silver",
    "refresh_gold",
]

missing_functions = [
    name
    for name in required_functions
    if not callable(globals().get(name))
]

if missing_functions:
    raise RuntimeError(
        f"Required pipeline functions are missing: {missing_functions}"
    )

print("Bronze pipeline available: PASS")
print("Silver pipeline available: PASS")
print("Gold pipeline available: PASS")
print("End-to-end dependencies ready.")

%md
## 2. Define the End-to-End Pipeline
Process a selected raw batch through Bronze, Silver, and Gold.
Support restarting from existing Bronze or refreshing Gold directly.
Stop downstream processing if an upstream operation fails.
Record an overall execution log alongside individual operation logs.

In [0]:
def run_petroleum_pipeline(
    *,
    batch_id,
    load_type,
    start_layer="raw",
    source_folder=None,
    backfill_mode=False,
):
    """Execute the pipeline from the requested starting layer."""
    pipeline_run_id = str(uuid4())
    start_time = datetime.now(timezone.utc)

    status = "Failure"
    error_message = None
    results = {}

    source_reference = json.dumps({
        "batch_id": batch_id,
        "load_type": load_type,
        "start_layer": start_layer,
        "source_folder": (
            str(source_folder) if source_folder is not None else None
        ),
        "backfill_mode": backfill_mode,
    })

    try:
        if not isinstance(batch_id, str) or not batch_id.strip():
            raise ValueError("Batch ID must be a nonempty string.")

        if load_type not in {"full_load", "incremental_load"}:
            raise ValueError("Unsupported load type.")

        if start_layer not in {"raw", "bronze", "silver"}:
            raise ValueError("Start layer must be raw, bronze, or silver.")

        if not isinstance(backfill_mode, bool):
            raise ValueError("backfill_mode must be True or False.")

        if start_layer == "raw":
            if source_folder is None or not str(source_folder).strip():
                raise ValueError("A source folder is required for raw ingestion.")

            print("Starting Raw-to-Bronze processing.")

            results["bronze"] = ingest_raw_batch_to_bronze(
                source_folder,
                batch_id=batch_id,
                load_type=load_type,
            )

        if start_layer in {"raw", "bronze"}:
            print("Starting Bronze-to-Silver processing.")

            results["silver"] = process_bronze_to_silver(
                batch_id=batch_id,
                load_type=load_type,
                backfill_mode=backfill_mode,
            )

        # Gold always reads current Silver, including protected revisions.
        print("Starting Silver-to-Gold processing.")

        results["gold"] = refresh_gold(
            batch_id=batch_id,
            load_type=load_type,
        )

        status = "Success"

    except Exception as exc:
        error_message = f"{type(exc).__name__}: {exc}"
        raise

    finally:
        end_time = datetime.now(timezone.utc)

        # Layer-specific row metrics remain in their individual audit entries.
        write_execution_log(
            run_id=pipeline_run_id,
            batch_id=batch_id,
            load_type=load_type,
            layer="End-to-End",
            source_reference=source_reference,
            target_table=gold_tables["fact"],
            start_time=start_time,
            end_time=end_time,
            status=status,
            error_message=error_message,
        )

    return {
        "pipeline_run_id": pipeline_run_id,
        "status": status,
        "duration_seconds": (end_time - start_time).total_seconds(),
        "results": results,
    }


print("End-to-end pipeline function ready.")

%md
## 3. Configure Execution Parameters
Select the saved batch, load type, starting layer, and backfill mode.
These parameters support incremental processing and historical reruns.

In [0]:
dbutils.widgets.text(
    "raw_root",
    "/Volumes/workspace/default/petroleum_raw/phase1_expanded",
)
dbutils.widgets.text("batch_id", "20261003T114452938244Z")

dbutils.widgets.dropdown(
    "load_type",
    "incremental_load",
    ["full_load", "incremental_load"],
)
dbutils.widgets.dropdown(
    "start_layer",
    "raw",
    ["raw", "bronze", "silver"],
)
dbutils.widgets.dropdown(
    "backfill_mode",
    "false",
    ["false", "true"],
)

pipeline_raw_root = dbutils.widgets.get("raw_root").strip()
pipeline_batch_id = dbutils.widgets.get("batch_id").strip()
pipeline_load_type = dbutils.widgets.get("load_type")
pipeline_start_layer = dbutils.widgets.get("start_layer")
pipeline_backfill_text = dbutils.widgets.get("backfill_mode")

if not pipeline_batch_id:
    raise ValueError("Batch ID must not be empty.")

if pipeline_backfill_text not in {"true", "false"}:
    raise ValueError("Backfill mode must be true or false.")

pipeline_backfill_mode = pipeline_backfill_text == "true"

pipeline_source_folder = (
    Path(pipeline_raw_root)
    / pipeline_batch_id
    / pipeline_load_type
)

if pipeline_start_layer == "raw":
    if not pipeline_source_folder.is_dir():
        raise FileNotFoundError(
            f"Source folder not found: {pipeline_source_folder}"
        )

    selected_files = sorted(
        pipeline_source_folder.glob("page_*.json")
    )

    if not selected_files:
        raise FileNotFoundError("No JSON pages found in the selected folder.")

    print(f"Source folder: {pipeline_source_folder}")
    print(f"Files selected: {len(selected_files)}")

print(f"Batch ID: {pipeline_batch_id}")
print(f"Load type: {pipeline_load_type}")
print(f"Start layer: {pipeline_start_layer}")
print(f"Backfill mode: {pipeline_backfill_mode}")
print("Execution parameters ready.")

%md
## 4. Execute the Pipeline
Run the selected batch through the configured processing stages.
Refresh Gold automatically after successful Silver processing.

In [0]:
pipeline_result = run_petroleum_pipeline(
    batch_id=pipeline_batch_id,
    load_type=pipeline_load_type,
    start_layer=pipeline_start_layer,
    source_folder=pipeline_source_folder,
    backfill_mode=pipeline_backfill_mode,
)

print("\nPipeline execution summary")
print(f"Pipeline run ID: {pipeline_result['pipeline_run_id']}")
print(f"Status: {pipeline_result['status']}")
print(
    f"Total duration: "
    f"{pipeline_result['duration_seconds']:.2f} seconds"
)

stage_results = pipeline_result["results"]

if "bronze" in stage_results:
    print(
        f"Bronze inserts: "
        f"{stage_results['bronze']['rows_inserted']:,}"
    )

if "silver" in stage_results:
    print(
        f"Silver inserts: "
        f"{stage_results['silver']['rows_inserted']:,}"
    )
    print(
        f"Silver updates: "
        f"{stage_results['silver']['rows_updated']:,}"
    )

gold_fact_result = stage_results["gold"]["fact_result"]

print(f"Gold Fact inserts: {gold_fact_result['rows_inserted']:,}")
print(f"Gold Fact updates: {gold_fact_result['rows_updated']:,}")

%md
## 5. Verify End-to-End Audit Coverage
Verify successful operation logs for the completed pipeline run.
Identify Silver and Gold operations by their returned run IDs.
Identify raw-file operations by batch, load type, and execution window.

In [0]:
audit_df = spark.table(execution_log_table)

overall_logs = audit_df.filter(
    (F.col("run_id") == pipeline_result["pipeline_run_id"])
    & (F.col("layer") == "End-to-End")
)

overall_rows = overall_logs.collect()

if len(overall_rows) != 1:
    raise AssertionError("Expected one overall pipeline audit entry.")

overall = overall_rows[0]
run_parameters = json.loads(overall["source_reference"])
stages = pipeline_result["results"]

# Select exact Silver and Gold operation IDs returned by the pipeline.
operation_ids = [
    result["run_id"]
    for result in stages["gold"]["dimension_results"]
]
operation_ids.append(stages["gold"]["fact_result"]["run_id"])
operation_ids.append(stages["gold"]["agg_result"]["run_id"])

if "silver" in stages:
    operation_ids.append(stages["silver"]["run_id"])

operation_logs = audit_df.filter(F.col("run_id").isin(operation_ids))
operation_rows = operation_logs.collect()

if (
    len(operation_rows) != len(operation_ids)
    or {row["run_id"] for row in operation_rows} != set(operation_ids)
):
    raise AssertionError("Missing or duplicate Silver/Gold audit entries.")

all_run_logs = overall_logs.unionByName(operation_logs)
expected_log_count = 1 + len(operation_ids)

if "bronze" in stages:
    # The batch wrapper returns counts, not individual file run IDs.
    # This lookup assumes no overlapping run of the same batch/load type.
    raw_logs = audit_df.filter(
        (F.col("layer") == "Raw-to-Bronze")
        & (F.col("batch_id") == overall["batch_id"])
        & (F.col("load_type") == overall["load_type"])
        & (F.col("start_time") >= F.lit(overall["start_time"]))
        & (F.col("end_time") <= F.lit(overall["end_time"]))
    )

    expected_files = stages["bronze"]["files_found"]

    if raw_logs.count() != expected_files:
        raise AssertionError(
            "Raw audit count mismatch. Check for missing logs "
            "or overlapping executions of the same batch."
        )

    all_run_logs = all_run_logs.unionByName(raw_logs)
    expected_log_count += expected_files

verified_rows = all_run_logs.collect()

for row in verified_rows:
    if row["status"] != "Success":
        raise AssertionError(f"Unsuccessful operation: {row['layer']}")

    if (
        row["start_time"] is None
        or row["end_time"] is None
        or row["end_time"] < row["start_time"]
    ):
        raise AssertionError("Missing or invalid execution timestamps.")

    if row["layer"] != "End-to-End":
        for metric in ["rows_read", "rows_inserted", "rows_updated"]:
            if row[metric] is None or row[metric] < 0:
                raise AssertionError(
                    f"Invalid {metric} for {row['target_table']}"
                )

display(
    all_run_logs.select(
        "layer",
        "target_table",
        "status",
        "rows_read",
        "rows_output",
        "rows_inserted",
        "rows_updated",
        (
            F.col("end_time").cast("double")
            - F.col("start_time").cast("double")
        ).alias("duration_seconds"),
        "start_time",
    ).orderBy("start_time")
)

print(f"Verified audit entries: {len(verified_rows)}")
print(f"Expected audit entries: {expected_log_count}")
print("End-to-end audit coverage: PASS")

%md
## 6. Export Verified Audit Evidence
Export the verified execution logs for the completed pipeline run.
Preserve operation metrics and timings for reporting and FinOps analysis.

In [0]:
import csv

evidence_folder = Path(
    "/Volumes/workspace/default/petroleum_raw/phase3_evidence"
)
evidence_folder.mkdir(parents=True, exist_ok=True)

evidence_file = (
    evidence_folder
    / f"pipeline_audit_{pipeline_result['pipeline_run_id']}.csv"
)

evidence_columns = [
    "run_id",
    "batch_id",
    "load_type",
    "layer",
    "source_reference",
    "target_table",
    "status",
    "rows_read",
    "rows_output",
    "rows_aggregated",
    "rows_inserted",
    "rows_updated",
    "rows_quarantined",
    "start_time",
    "end_time",
    "duration_seconds",
    "error_message",
]

evidence_df = (
    all_run_logs
    .withColumn(
        "duration_seconds",
        F.col("end_time").cast("double")
        - F.col("start_time").cast("double"),
    )
    .orderBy("start_time")
    .select(*evidence_columns)
)

# Only the small, verified audit result is collected.
evidence_rows = evidence_df.collect()

with evidence_file.open("w", encoding="utf-8", newline="") as destination:
    writer = csv.DictWriter(
        destination,
        fieldnames=evidence_columns,
    )
    writer.writeheader()
    writer.writerows(row.asDict() for row in evidence_rows)

print(f"Audit entries exported: {len(evidence_rows)}")
print(f"Evidence file: {evidence_file}")
print("Timestamp timezone: UTC")

In [0]:
   fact_plan_df = prepare_gold_fact(spark.table(silver_table))
   fact_plan_df.explain("formatted")
   print(f"Prepared fact rows: {fact_plan_df.count():,}")